# 第6章　モデルを現場に届ける ― 軽量化・書き出し・推論の最適化

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 6.1　モデルを書き出す ― 学習の外へ持ち出す

In [ ]:
import torch
model.eval()
dummy = torch.randn(1, 3, 224, 224)          # 入力の形を一つ示す
torch.onnx.export(model, dummy, "model.onnx",  # ONNX形式で書き出す
                  input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}})   # バッチサイズは可変に

## ONNX量子化を実装し、性能を測り直す

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic("model.onnx", "model.int8.onnx", weight_type=QuantType.QInt8)

## レイテンシとスループットを実測する

In [ ]:
import numpy as np
import time, numpy as np, onnxruntime as ort
sess = ort.InferenceSession("model.onnx", providers=["CUDAExecutionProvider"])
x = np.random.randn(1, 3, 224, 224).astype(np.float32)
for _ in range(10):                      # ウォームアップ（初回は遅い）
    sess.run(None, {"image": x})
lat = []
for _ in range(200):
    t = time.perf_counter(); sess.run(None, {"image": x}); lat.append(time.perf_counter() - t)
lat = np.array(lat) * 1000
print(f"p50={np.percentile(lat,50):.1f}ms  p95={np.percentile(lat,95):.1f}ms")

## 負荷試験でキャパシティを裏づける

In [ ]:
# locust の骨格：検査到着を模して負荷をかける
from locust import HttpUser, task, constant_throughput
class Radiology(HttpUser):
    wait_time = constant_throughput(2)   # 1ユーザーあたり毎秒2件相当
    @task
    def infer(self):
        self.client.post("/infer", files={"dicom": DUMMY_STUDY})
# 実行例: locust -u 50 -r 10 --run-time 10m  （同時50、毎秒10ずつ増やす）

## キャッシュと冪等性で無駄打ちを減らす

```text
入力manifest = { 施設・テナント, 対象タスク, 選択したシリーズとSOPインスタンスの集合,
                 （必要なら）画像内容のハッシュ }
処理manifest = { 前処理版, モデル版, 後処理版, 閾値設定の版 }
キャッシュキー = hash(入力manifest + 処理manifest)
値 = { 推論結果, 確信度, 生成時刻, 各版 }   # TTLは臨床の再読影サイクルに合わせる
```

## データドリフトを数値で検知する

In [ ]:
import numpy as np
def psi(expected, actual, bins=10):
    q = np.quantile(expected, np.linspace(0, 1, bins + 1))
    # 確信度が0付近に張り付くと分位点が同じ値だらけになり、ビン境界が重複して
    # どんなドリフトもPSI=0になる。重複を潰し、潰れすぎたら等幅ビンに切り替える。
    q = np.unique(q)
    if len(q) - 1 < max(2, bins // 2):
        lo = min(float(np.min(expected)), float(np.min(actual)))
        hi = max(float(np.max(expected)), float(np.max(actual)))
        q = np.linspace(lo, hi, bins + 1)
    q[0], q[-1] = -np.inf, np.inf
    e = np.histogram(expected, q)[0] / len(expected) + 1e-6
    a = np.histogram(actual,   q)[0] / len(actual)   + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

## 可観測性 ― ログ・メトリクス・トレースで内部を見える化する

```json
{"時刻":"2026-07-05T02:11:04Z","相関ID":"study-8f3a…","段階":"推論",
 "モデル版":"v3.3.0","レイテンシms":21840,"結果":"陽性候補1","確信度":0.82,
 "レベル":"INFO"}
```

## エラーバジェットを、運用の規律に変える ― バーンレート警報

```text
重大(即ページ) : 直近1時間で14.4倍 かつ 直近5分でも14.4倍を超える
                 → 2日でバジェット全消費のペース。本物の急性障害だけが両窓を満たす
警告(翌営業日) : 直近6時間で6倍   かつ 直近30分でも6倍を超える
                 → 緩慢な悪化。夜中に人を起こさず、日中に調べる
```

## シンセティック監視 ― 「壊れた沈黙」を、待たずに捕まえる

In [ ]:
# 数分おきに「答えの分かっている検査」を流し、結果を検証する（擬似コード）
def synthetic_probe():
    r = pipeline.infer(GOLDEN_STUDY)                 # 合成ファントム検査を投入
    assert r.status == "ok"
    assert abs(r.score - GOLDEN_EXPECTED) < 0.02      # 既知の期待スコアと一致するか
    assert r.latency_ms < SLO_LATENCY                 # 端から端までの所要時間
    emit_metric("synthetic_ok", 1)                    # 監視盤・アラートへ

## 新版への切り替えを、段階で行う ― カナリアとA/Bテスト

```yaml
# 推論ルーターの振り分け設定（施設単位のカナリア）
routing:
  model_stable:   { version: v3.2.0, weight: 90 }   # 既存の安定版
  model_canary:   { version: v3.3.0, weight: 10 }    # 新版（まず一部施設のみ）
  assignment_key: institution_id     # 検査単位でなく施設単位で固定割当
  sticky: true                       # 同一施設は常に同じ版へ（結果の一貫性）
guardrails:            # この条件に触れたら自動でカナリアを停止（weightを0へ）
  canary_error_rate_max: 0.005       # 推論失敗率の上限
  canary_p95_latency_ms: 30000       # p95レイテンシの上限
  canary_positive_rate_delta: 0.05   # 陽性率が安定版から乖離したら停止
```

## パイプライン全体像と、Blue-Greenという切り替え方

```text
コミット → ①ビルド・テスト(第2章) → ②モデルをレジストリへ登録＋署名
        → ③ステージング環境へ自動デプロイ → ④回帰試験セットで性能ゲート
        → ⑤（人間の承認：臨床＋品質）→ ⑥本番へBlue-Green/カナリアで反映
```

## 推論をスケールさせ、コストを抑える

```yaml
# キュー長に応じてGPUワーカーを増減させる（Kubernetes HPAの例）
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
spec:
  minReplicas: 1        # 0にしない：モデルロードに数十秒かかり、救急で致命的
  maxReplicas: 8
  metrics:
    - type: External
      external:
        metric: { name: inference_queue_depth }   # 待ち件数（Prometheus等から）
        target: { type: AverageValue, averageValue: "5" }  # 1ワーカーあたり5件を目安
```

```text
1検査あたりGPUコスト = (GPU単価[円/時] ÷ 3600) × 1検査のGPU秒 ÷ GPU利用率
例：A100が400円/時、1検査に2GPU秒、平均利用率40%
  = (400/3600) × 2 ÷ 0.40 ≈ 0.56円/検査
```

## オンプレGPUを、実際に運用する

```bash
nvidia-smi -mig 1                       # MIGモードを有効化
nvidia-smi mig -cgi 1g.10gb -C          # 対応GPUごとに、指定プロファイルのインスタンスを1個作成
```

## 特徴量とメタデータを、再利用する ― 軽量な特徴量ストア

```text
検査UID | 撮影日時 | 前処理版 | モデル版 | embedding(vector) | メタ特徴(JSON) | 有効期間
```